# Explain RAG with LangChain

**RAG = retrieve useful information, then give it to a language model to answer a question.**

We will use the same fictional university handbook and build the full pipeline:

**Documents → Chunks → Embeddings → Vector store → Retrieval → Prompt → Answer**


## 1. Setup — run once before presenting



In [ ]:
%pip -q install --upgrade "langchain-core>=1,<2" "langchain-text-splitters>=1,<2" "langchain-huggingface>=1,<2" "sentence-transformers>=5.2,<6" "transformers==5.17.0" "huggingface-hub>=1.23.0,<2" "sentencepiece>=0.2"


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
embedding_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small", use_safetensors=True
).eval()
print("Models ready.")


## 2. Load the documents


In [ ]:
handbook = {
    'library.txt': """The Aster University library opens at 8:30 AM and closes at 8:00 PM from Monday to Friday. On Saturday it opens at 9:00 AM and closes at 1:00 PM. It is closed on Sunday.

Undergraduate students can borrow up to four books for fourteen days. A loan can be renewed once if nobody has reserved the book. Reference books must stay inside the library.

Quiet study rooms are on the second floor. Groups book discussion rooms through the library desk. Food is not allowed near the library computers.""",

    'network.txt': """The student Wi-Fi network is called Aster-Student. Students sign in with their university email address and campus password. Visitors use the separate Aster-Guest network.

To reset a forgotten campus password, visit the IT help desk in room B14 with a student ID card. The help desk opens from 9:00 AM to 4:00 PM on weekdays. Never share your campus password.

Report Wi-Fi connection problems to the IT help desk. Include the building name and the time of the problem. The library has wired computers for students who cannot connect their laptops.""",

    'robotics.txt': """The robotics laboratory is in room C207. Students must complete a safety induction before using any robot. Closed shoes are required during practical work.

A robotics laboratory session must be booked at least 48 hours in advance using the student portal. Each booking lasts at most two hours. A laboratory supervisor must be present during the session.

Food and drinks are prohibited in the robotics laboratory. Press the red emergency stop button if a robot moves unexpectedly. Report damaged equipment to the supervisor before starting work.""",

    'assessment.txt': """The Applied AI course awards 40 percent of its final grade for the project and 60 percent for the examination. The minimum overall passing mark is 50 percent.

The project deadline is 18 November at 5:00 PM. Submit one PDF report and one notebook through the course portal. The report should describe the dataset, method, experiments, and limitations.

An extension request must be submitted before the project deadline. Include the reason and supporting evidence. The course coordinator reviews requests and sends the decision through the portal.""",

    'shuttle.txt': """The Aster campus shuttle leaves the main gate every 30 minutes between 7:00 AM and 6:00 PM on weekdays. There is no shuttle service on weekends.

The shuttle stops at the main gate, science building, library, and student residence. Students travel free when they show a valid student ID card. Visitors must arrange their own transport.

The last shuttle departs the main gate at 6:00 PM. Lost property from the shuttle is taken to the campus security office the following morning.""",

    'printing.txt': """Students receive 120 free black-and-white printing pages each semester. Colour printing is not included in this allowance. Unused free pages do not carry over to the next semester.

Printers are located in the library and the science building lobby. Use your student ID card to release a print job. Jobs that are not released within 24 hours are deleted from the print queue.

Report printer faults to the IT help desk. Do not open a printer to fix a paper jam yourself. The help desk can review a failed print job if your page allowance was deducted.""",

}

documents = [
    Document(page_content=text, metadata={"source": name})
    for name, text in handbook.items()
]
print("Documents:", len(documents))


## 3. Split into chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    embedding_tokenizer, chunk_size=80, chunk_overlap=16
)
chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))
print("Example source:", chunks[0].metadata["source"])
print(chunks[0].page_content)


## 4. Embed and store


In [ ]:
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Ready to retrieve the three most relevant chunks.")


## 5. Retrieve evidence


In [ ]:
question = "How early should I book the robotics laboratory?"
retrieved_docs = retriever.invoke(question)

for doc in retrieved_docs:
    print("\nSource:", doc.metadata["source"])
    print(doc.page_content)


## 6. Connect the prompt to the model


In [ ]:
prompt = PromptTemplate.from_template(
    "Answer using only the context. Answer every part of the question briefly. "
    "If the context lacks the answer, say 'I do not know from the documents.'\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

@torch.inference_mode()
def generate(prompt_value):
    inputs = tokenizer(prompt_value.to_string(), return_tensors="pt")
    if inputs["input_ids"].shape[1] > 512:
        raise ValueError("Prompt too long: use fewer/smaller chunks or a shorter question.")
    output = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)

answer_chain = prompt | RunnableLambda(generate)


## 7. Put it together


In [ ]:
def ask(question, search=retriever, qa=answer_chain):
    evidence = search.invoke(question)
    context = "\n\n".join(doc.page_content for doc in evidence)
    answer = qa.invoke({"context": context, "question": question})
    return answer, evidence

answer, evidence = ask(question)
print("Question:", question)
print("Answer:", answer)
print("Sources supplied:", [doc.metadata["source"] for doc in evidence])




## 8. Compare with and without retrieval
**Say:** “The difference is the evidence we supply, not additional model training.”

We use the same question and instructions for both versions. With no context, an honest model
should decline to answer about this fictional university; it may still guess.


In [ ]:
print("Without retrieval:", answer_chain.invoke({"context": "", "question": question}))
print("With retrieval:   ", ask(question)[0])


## 9. Check retrieval separately from the answer
**Say:** “Finding the right document and writing a correct answer are different tasks.”

For one question, inspect both. A source match alone does not prove the exact answer sentence was
retrieved. A correct passage can still lead to an incomplete answer.


In [ ]:
test_question = "How many books can an undergraduate borrow and for how long?"
test_answer, test_evidence = ask(test_question)

print("Answer:", test_answer)
print("Expected facts: four books; fourteen days.")
print("Expected source retrieved:", any(d.metadata["source"] == "library.txt" for d in test_evidence))
for doc in test_evidence:
    print("\n", doc.metadata["source"], "—", doc.page_content)


## 10. Challenge: improve the assistant's reliability
**Same goal as the original challenge:** improve incomplete or unsupported answers and compare
with the baseline. Keep the handbook and models unchanged.

1. Test the **four development questions** below. Identify one failure using the answer and evidence.
2. Make **one improvement**: a clearer prompt, different retrieval settings, or separate subquestions
   for a multi-part question. Create new objects so the baseline remains unchanged.
3. Compare baseline and improved answers on the same questions. Then freeze the change and test
   the two held-out questions once. If you tune on those, write two new held-out questions.
4. Explain what improved, what did not, and why. A well-explained negative result still counts.

Do not hard-code answers or put the expected facts into the generation prompt.

| Development question | Expected facts / behavior |
|---|---|
| What must I complete before using a robot, and how early must I book? | Safety induction; at least 48 hours in advance |
| How many books can an undergraduate borrow and for how long? | Four books; fourteen days |
| Where is the robotics laboratory, and what is the student Wi-Fi network called? | Room C207; Aster-Student |
| What is the name of the library director? | Explicitly state that the documents do not provide the name |

### A short place to implement your improvement
The starter deliberately returns the baseline unchanged. Replace its body after diagnosing a failure.
Return `(answer, evidence)` like `ask`; if you use multiple model calls, include all evidence actually
supplied to those calls. Do not change the original `retriever`, `prompt`, `answer_chain`, or `ask`.


In [ ]:
def improved_ask(question):
    # Replace this line with your improvement.
    return ask(question)

# Change this to any development question from the table.
challenge_question = "What must I complete before using a robot, and how early must I book?"

for label, method in [("Baseline", ask), ("Improved", improved_ask)]:
    answer, evidence = method(challenge_question)
    print("\n" + label + ":", answer)
    for doc in evidence:
        print(doc.metadata["source"], "—", doc.page_content)


### Review the answers
For each question, write one row in your notes:

| Question | Version | Complete? | Supported by supplied evidence? | Correctly answered or declined? |
|---|---|---|---|---|
| … | Baseline / improved | Yes / no | Yes / no | Brief explanation |

Refusing every question is not success. Check that the assistant answers answerable questions and
admits when information is missing. Keep development and held-out results separate. This tiny test
set is for learning, not proof of general accuracy.

**Held-out questions — use after freezing your change:**
- “At what time does the library close on Saturday, and does the campus shuttle run on weekends?”
  Expected: **1:00 PM; no weekend shuttle service**.
- “What is the cost of a replacement student ID card?”
  Expected: **the documents do not provide the cost**.

**Submit:** your code change, comparison notes, and a 100–150 word explanation.

**Optional 10-mark rubric:** diagnosis with evidence (2); working change without hard-coded answers (3);
fair comparison and held-out review (3); explanation of limitations (2).

<details><summary>Hints and optional experiments</summary>

- If the prompt contains both facts but the answer mentions only one, inspect generation, not just retrieval.
- For a new prompt, create `new_prompt`, then `new_chain = new_prompt | RunnableLambda(generate)`.
  Use `ask(question, qa=new_chain)` inside `improved_ask`.
- For retrieval changes, create `new_retriever` and pass `search=new_retriever`.
  Try `vector_store.as_retriever(search_kwargs={"k": 2})` or MMR retrieval with
  `search_type="mmr", search_kwargs={"k": 3, "fetch_k": 8}`.
- To explore chunking, create a new splitter and rebuild a separate vector store. Keep chunks at
  most 200 MiniLM tokens and ensure the full prompt fits the 512-token check.
- To add knowledge, append a new `Document`, split it with the other documents, and build a new
  store and retriever. The model weights do not change.

</details>